# Testing Notebook

Notebook này chỉ dùng cho tập kiểm thử. Không có bước `fit` hoặc `fit_transform` trên test.


## Kiểm soát rò rỉ dữ liệu

- Chỉ đọc `raw_data_test.csv`.
- Chỉ load model đã huấn luyện từ `training.ipynb`.
- Trên test chỉ được dùng `transform` và `predict`.


> Kết quả suy diễn trên tập kiểm thử được trình bày trực tiếp trong notebook để thuận tiện cho việc kiểm tra và đối chiếu khi nộp bài.


In [1]:
from __future__ import annotations

import math
import re
import unicodedata
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import normalize


In [2]:
BASE_DIR = Path(".")
RAW_TEST_PATH = BASE_DIR / "raw_data_test.csv"
CLEAN_TEST_PATH = BASE_DIR / "clean_data_test.csv"

ARTIFACTS_DIR = BASE_DIR / "artifacts"
CLEAN_DIR = ARTIFACTS_DIR / "clean"
FEATURES_DIR = ARTIFACTS_DIR / "features"
CLUSTERING_DIR = ARTIFACTS_DIR / "clustering"
FIGURES_DIR = CLUSTERING_DIR / "figures"

RANDOM_STATE = 42
TRANSFORM_CHUNK_SIZE = 50000

for path in [CLEAN_DIR, FEATURES_DIR, CLUSTERING_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## 1. Hàm tiện ích

In [3]:
TEXT_COLUMNS = [
    "job_title",
    "company_name",
    "location",
    "job_type",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
    "job_description",
    "benefits",
    "requirements",
]
TEXT_COLUMNS_FOR_CLUSTERING = ["job_title", "job_description", "requirements", "benefits"]
CLUSTER_LABEL_MAP = {
    0: "IT - phần mềm - kinh doanh/marketing chuyên môn",
    1: "Kỹ thuật - sản xuất - bảo trì",
    2: "Kinh doanh - bán hàng - chăm sóc khách hàng",
    3: "Kho vận - giao nhận - lao động phổ thông",
    4: "Marketing - nội dung - thiết kế",
    5: "Hành chính nhân sự - văn phòng - xuất nhập khẩu",
    6: "Bán lẻ - dịch vụ - lao động phổ thông",
    7: "Tư vấn giáo dục - quản lý trung tâm",
    8: "Kế toán - kiểm toán - tài chính doanh nghiệp",
    9: "Tài chính - ngân hàng - thu hồi/nhắc phí",
    10: "Xây dựng - kiến trúc - kỹ thuật công trình",
    11: "Telesales - tư vấn qua điện thoại",
    12: "Giáo dục - đào tạo - giáo viên ngoại ngữ",
}

AMBIGUOUS_SALARY_KEYWORDS = [
    "thoa thuan", "thương lượng", "thuong luong", "canh tranh", "cạnh tranh", "negotiable",
    "deal luong", "dang cap nhat", "đang cập nhật", "updating", "update", "competitive",
]
URL_REGEX = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_REGEX = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", flags=re.IGNORECASE)
PHONE_REGEX = re.compile(r"(?<!\d)(?:\+?84|0)(?:[\s\.-]?\d){8,10}(?!\d)")
HTML_REGEX = re.compile(r"<[^>]+>")
WHITESPACE_REGEX = re.compile(r"\s+")
REPEATED_PUNCT_REGEX = re.compile(r"([!?,.;:\-_=+/\\|])\1+")
NUMBER_TOKEN_REGEX = re.compile(r"\d[\d\.,]*")
RANGE_SPLIT_REGEX = re.compile(r"\s*(?:\-|~|to|den|t[ơo]i|–|—)\s*", flags=re.IGNORECASE)
MILLION_REGEX = re.compile(r"\b(?:trieu|triệu)\b|\btr\b|(?<=\d)(?:tr|trieu|triệu)\b", re.IGNORECASE)
VND_UNIT_REGEX = re.compile(r"\b(?:vnd|vnđ|dong)\b|(?<!\w)đ(?!\w)", re.IGNORECASE)


def ensure_input_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file đầu vào: {path}. "
            "Hãy đặt raw_data_test.csv ở root project trước khi chạy testing.ipynb."
        )


def normalize_unicode_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return unicodedata.normalize("NFKC", str(value))


def clean_text_basic(value: Any) -> str:
    text = normalize_unicode_text(value).lower().strip()
    text = HTML_REGEX.sub(" ", text)
    text = URL_REGEX.sub(" ", text)
    text = EMAIL_REGEX.sub(" ", text)
    text = PHONE_REGEX.sub(" ", text)
    text = REPEATED_PUNCT_REGEX.sub(r"\1", text)
    text = WHITESPACE_REGEX.sub(" ", text)
    return text.strip()


def clean_salary_text_basic(value: Any) -> str:
    text = normalize_unicode_text(value).lower().strip()
    text = HTML_REGEX.sub(" ", text)
    text = URL_REGEX.sub(" ", text)
    text = EMAIL_REGEX.sub(" ", text)
    text = REPEATED_PUNCT_REGEX.sub(r"\1", text)
    text = WHITESPACE_REGEX.sub(" ", text)
    return text.strip()


def fold_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")


def normalize_salary_text(value: Any) -> str:
    text = fold_accents(clean_salary_text_basic(value))
    replacements = {
        "us$": "usd",
        "$/month": " usd ",
        "$ /month": " usd ",
        "$/thang": " usd ",
        "/month": " / month ",
        "/thang": " / month ",
        "tri u": "trieu",
        "trieu dong": "trieu",
        "vnd/thang": "vnd",
        "vnđ/tháng": "vnd",
    }
    for source, target in replacements.items():
        text = text.replace(source, target)
    text = text.replace("~", " - ").replace("–", " - ").replace("—", " - ")
    return WHITESPACE_REGEX.sub(" ", text).strip()


def classify_salary_pattern(text: str) -> str:
    if not text:
        return "missing"
    if any(keyword in text for keyword in AMBIGUOUS_SALARY_KEYWORDS):
        return "ambiguous"
    has_range = bool(RANGE_SPLIT_REGEX.search(text))
    has_number = bool(NUMBER_TOKEN_REGEX.search(text))
    has_usd = "usd" in text or "$" in text
    has_vnd = bool(MILLION_REGEX.search(text) or VND_UNIT_REGEX.search(text))
    if has_range and has_number:
        return "range_usd" if has_usd else "range_vnd" if has_vnd else "range_unknown"
    if has_number:
        return "single_usd" if has_usd else "single_vnd" if has_vnd else "single_unknown"
    return "invalid"


def parse_number_token(token: str, currency_hint: str) -> float | None:
    cleaned = token.strip()
    if not cleaned:
        return None
    if currency_hint == "usd":
        cleaned = cleaned.replace(",", "")
        if cleaned.count(".") > 1:
            cleaned = cleaned.replace(".", "")
        try:
            return float(cleaned)
        except ValueError:
            return None
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", cleaned):
        return float(cleaned.replace(".", ""))
    if re.fullmatch(r"\d{1,3}(,\d{3})+", cleaned):
        return float(cleaned.replace(",", ""))
    if cleaned.count(",") == 1 and cleaned.count(".") == 0:
        left, right = cleaned.split(",")
        if len(right) <= 2:
            cleaned = f"{left}.{right}"
    try:
        return float(cleaned.replace(",", ""))
    except ValueError:
        return None


def infer_currency(text: str) -> str | None:
    if "usd" in text or "$" in text:
        return "usd"
    if MILLION_REGEX.search(text) or VND_UNIT_REGEX.search(text):
        return "vnd"
    return None


def convert_to_million_vnd(value: float, currency: str, text: str, usd_to_vnd: float = 25000.0) -> float:
    if currency == "usd":
        return value * (usd_to_vnd / 1_000_000.0)
    if MILLION_REGEX.search(text):
        return value
    if VND_UNIT_REGEX.search(text):
        return value / 1_000_000.0
    return value


def parse_salary_row(raw_salary: Any) -> dict[str, Any]:
    original_text = normalize_unicode_text(raw_salary)
    normalized = normalize_salary_text(original_text)
    pattern = classify_salary_pattern(normalized)
    result = {
        "salary_raw_normalized": normalized,
        "salary_pattern": pattern,
        "salary_currency": None,
        "salary_min": np.nan,
        "salary_max": np.nan,
        "salary_expected_million_vnd": np.nan,
        "salary_parse_status": "invalid",
    }
    if pattern in {"missing", "ambiguous", "invalid"}:
        result["salary_parse_status"] = pattern
        return result
    currency = infer_currency(normalized)
    if currency is None:
        result["salary_parse_status"] = "unknown_currency"
        return result
    numbers = [parse_number_token(token, currency) for token in NUMBER_TOKEN_REGEX.findall(normalized)]
    numbers = [number for number in numbers if number is not None]
    if not numbers:
        result["salary_parse_status"] = "no_numbers"
        return result
    has_range = len(RANGE_SPLIT_REGEX.split(normalized)) > 1 and len(numbers) >= 2
    raw_min, raw_max = sorted(numbers[:2]) if has_range else (numbers[0], numbers[0])
    salary_min = convert_to_million_vnd(raw_min, currency, normalized)
    salary_max = convert_to_million_vnd(raw_max, currency, normalized)
    result.update(
        {
            "salary_currency": currency,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "salary_expected_million_vnd": (salary_min + salary_max) / 2.0,
            "salary_parse_status": "valid",
        }
    )
    return result


def ensure_base_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    for column in TEXT_COLUMNS + ["salary"]:
        if column not in result.columns:
            result[column] = "unknown"
    if "id" not in result.columns:
        result["id"] = np.arange(1, len(result) + 1)
    return result


def simplify_location(value: object) -> str:
    text = "" if pd.isna(value) else fold_accents(str(value).lower())
    mapping = [
        ("ho chi minh", ["ho chi minh", "tphcm", "tp.hcm", "hcm", "sai gon"]),
        ("ha noi", ["ha noi", "hn"]),
        ("da nang", ["da nang"]),
        ("binh duong", ["binh duong"]),
        ("dong nai", ["dong nai"]),
        ("hai phong", ["hai phong"]),
        ("can tho", ["can tho"]),
        ("bac ninh", ["bac ninh"]),
    ]
    for label, keywords in mapping:
        if any(keyword in text for keyword in keywords):
            return label
    if text.strip() in {"", "unknown", "khong"}:
        return "unknown"
    return "other"


def make_final_text(df: pd.DataFrame) -> pd.Series:
    parts = []
    for column in TEXT_COLUMNS_FOR_CLUSTERING:
        value = df[column].fillna("").astype(str)
        if column == "job_title":
            value = value + " " + value
        parts.append(value)
    return pd.concat(parts, axis=1).agg(" ".join, axis=1).str.replace(r"\s+", " ", regex=True).str.strip()


def prepare_clean_data(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = ensure_base_columns(df)
    parsed = cleaned["salary"].apply(parse_salary_row).apply(pd.Series)
    cleaned = pd.concat([cleaned, parsed], axis=1)
    cleaned["salary_range_width"] = cleaned["salary_max"] - cleaned["salary_min"]
    cleaned = cleaned.loc[
        (cleaned["salary_parse_status"] == "valid")
        & cleaned["salary_expected_million_vnd"].notna()
        & (cleaned["salary_expected_million_vnd"] > 0)
        & (cleaned["salary_min"] <= cleaned["salary_max"])
    ].copy()
    for column in TEXT_COLUMNS:
        cleaned[column] = cleaned[column].map(clean_text_basic).replace({"": "unknown", "khong": "unknown", "none": "unknown", "n/a": "unknown", "null": "unknown"})
    cleaned["location_simplified"] = cleaned["location"].map(simplify_location)
    cleaned["is_salary_outlier"] = False
    cleaned["final_text"] = make_final_text(cleaned)
    return cleaned.reset_index(drop=True)


def transform_in_chunks(estimator, X: np.ndarray, chunk_size: int = 50000) -> np.ndarray:
    if X.shape[0] <= chunk_size:
        return estimator.transform(X)
    chunks = []
    for start in range(0, X.shape[0], chunk_size):
        stop = min(start + chunk_size, X.shape[0])
        chunks.append(estimator.transform(X[start:stop]))
    return np.vstack(chunks)


def save_plot(fig: plt.Figure, path: Path) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def load_cluster_label_map() -> dict[int, str]:
    profile_path = CLUSTERING_DIR / "final_cluster_profiles.csv"
    if profile_path.exists():
        profile_df = pd.read_csv(profile_path)
        if {"cluster_id", "final_cluster_label"}.issubset(profile_df.columns):
            return dict(zip(profile_df["cluster_id"].astype(int), profile_df["final_cluster_label"].astype(str)))
    return CLUSTER_LABEL_MAP


def plot_test_cluster_distribution(test_clustered: pd.DataFrame, filename: str) -> None:
    counts = test_clustered["cluster_id"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([f"C{idx}" for idx in counts.index], counts.values, color="#6c8ebf")
    ax.set_title("Phân phối cụm trên tập kiểm thử")
    ax.set_xlabel("Cụm")
    ax.set_ylabel("Số lượng mẫu")
    save_plot(fig, FIGURES_DIR / filename)


def plot_test_salary_boxplot(test_clustered: pd.DataFrame, filename: str) -> None:
    if "salary_expected_million_vnd" not in test_clustered.columns:
        return
    grouped = [group["salary_expected_million_vnd"].dropna().values for _, group in test_clustered.groupby("cluster_id", sort=True)]
    if not grouped:
        return
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.boxplot(grouped, labels=[f"C{cluster_id}" for cluster_id in sorted(test_clustered["cluster_id"].unique())], showfliers=False)
    ax.set_title("Phân phối lương kỳ vọng trên tập kiểm thử theo cụm")
    ax.set_xlabel("Cụm")
    ax.set_ylabel("Triệu VND")
    save_plot(fig, FIGURES_DIR / filename)


## 2. Đọc dữ liệu test thô và model đã huấn luyện

In [4]:
ensure_input_file(RAW_TEST_PATH)
required_models = [
    FEATURES_DIR / "tfidf_model.joblib",
    FEATURES_DIR / "svd_model.joblib",
    CLUSTERING_DIR / "final_umap_model.joblib",
    CLUSTERING_DIR / "final_kmeans_model.joblib",
]
for path in required_models:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu model đã huấn luyện từ training.ipynb: {path}")

raw_test_df = pd.read_csv(RAW_TEST_PATH)
display(raw_test_df.head())
print("Số dòng raw test:", len(raw_test_df))


Tóm tắt dữ liệu đầu vào của quá trình suy diễn trên tập kiểm thử.


## 3. Làm sạch dữ liệu test

In [5]:
clean_test_df = prepare_clean_data(raw_test_df)
clean_test_df.to_csv(CLEAN_TEST_PATH, index=False)
clean_test_df.to_csv(CLEAN_DIR / "clean_data_test.csv", index=False)

display(clean_test_df.head())
print("Số dòng clean test:", len(clean_test_df))


Đã hoàn tất bước làm sạch dữ liệu kiểm thử và lưu tập dữ liệu sạch.


## 4. Transform và gán cụm cho test

In [6]:
tfidf = joblib.load(FEATURES_DIR / "tfidf_model.joblib")
svd = joblib.load(FEATURES_DIR / "svd_model.joblib")
reducer = joblib.load(CLUSTERING_DIR / "final_umap_model.joblib")
final_kmeans = joblib.load(CLUSTERING_DIR / "final_kmeans_model.joblib")
cluster_label_map = load_cluster_label_map()

X_test_features = tfidf.transform(clean_test_df["final_text"])
X_test_svd = svd.transform(X_test_features)
X_test_svd_norm = normalize(X_test_svd, norm="l2")
X_test_umap = transform_in_chunks(reducer, X_test_svd_norm, chunk_size=TRANSFORM_CHUNK_SIZE)
test_labels = final_kmeans.predict(X_test_umap)

final_test_clustered = clean_test_df.copy()
final_test_clustered["cluster_id"] = test_labels
final_test_clustered["cluster_label"] = final_test_clustered["cluster_id"].map(cluster_label_map)
final_test_clustered["final_cluster_label"] = final_test_clustered["cluster_label"]
final_test_clustered.to_csv(CLUSTERING_DIR / "final_test_clustered.csv", index=False)


id,job_title,company_name,location,location_simplified,job_industry,job_type,experience_level,education_level,job_position,salary_expected_million_vnd,salary_range_width,salary_min,salary_max,is_salary_outlier,final_text,cluster_id,final_cluster_label,cluster_label
369771,nhân viên phục vụ nhà hàng,"hoa viet joint venture corp., ltd","sân golf thủ đức - số 18, đường số 2, khu phố giãn dân, phường long thạnh mỹ | số 18 đường 2, khu phố 30, phường long bình, tp hồ chí minh, việt nam",other,lao động phổ thông,toàn thời gian,1 năm,không,nhân viên,5.35,0.1000000000000005,5.3,5.4,0,"nhân viên phục vụ nhà hàng nhân viên phục vụ nhà hàng nhận khách từ lễ tân của nhà hàng. - trình menu, tư vấn khách chọn món ăn, thức uống - ghi nhận thông tin order từ khách, xác nhận lại với khách chính xác tên món ăn , thức uống, số lượng, các yêu cầu đặc biệt. - chuyển order cho thu ngân, bếp, bar. - phục vụ món ăn, thức uống theo đúng order của khách - thời gian làm việc: xoay ca, mỗi ca 8 tiếng, off 1 ngày trong tuần (ca 1: 5h30; ca 2 11h; ca 3: 14h) - địa điểm làm việc: sân golf thủ đức - số 18 đường số 2, p.long thạnh mỹ, tp.thủ đức nữ, từ 20 tuổi trở lên - ngoại hình khá (cao >1m57) - tiếng anh giao tiếp cơ bản - tốt nghiệp từ trung cấp trở lên - có thể làm việc xoay ca lương thỏa thuận theo năng lực + phụ cấp cơm 30k/ngày + service charge(theo quý) + tiền tăng ca(nếu có) - cấp phát đồng phục - thưởng lễ tết, sinh nhật, hiếu hỉ; lương tháng 13. - hỗ trợ tiền vé xe/tàu về quê - chế độ nghỉ phép năm 14 ngày, bảo hiểm xã hội và bảo hiểm y tế theo quy định của pháp luật. - hợp đồng dài hạn sau thử việc",6,Bán lẻ - dịch vụ - lao động phổ thông,Bán lẻ - dịch vụ - lao động phổ thông
323832,nhân viên tư vấn bán hàng part-time,công ty tnhh thiết bị y tế thuận phong,"449/6 sư vạn hạnh, phường 12, | 449/6 sư vạn hạnh, phường 12, quận 10, tp hồ chí minh",other,bán hàng - kinh doanh,thực tập,2 năm,trung cấp,cộng tác viên,2.5,2.0,1.5,3.5,0,"nhân viên tư vấn bán hàng part-time nhân viên tư vấn bán hàng part-time giới thiệu, tư vấn cho khách hàng sản phẩm tại showroom/website - giữ gìn vệ sinh khu vực làm việc, showroom, đảm bảo không gian luôn sạch đẹp, ngăn nắp giới tính: nam/nữ, từ : 20 – 25 tuổi - trung thực, giao tiếp tốt, chịu khó học hỏi - nhiệt tình, năng động, có trách nhiệm trong công việc, có khả năng làm việc nhóm. - thời gian làm việc: xoay ca linh hoạt: ca 1: 8h30’- 12h00’ ca 2: 13h00’ – 17h30’ ca 3: 17h00’ - 21h00’ lương : 25.000/giờ - thử việc: 01 tháng - được cấp đồng phục làm việc - đồng nghiệp thân thiện, hòa đồng, vui vẻ - được tham gia các hoạt động ngoại khóa, team building công ty",6,Bán lẻ - dịch vụ - lao động phổ thông,Bán lẻ - dịch vụ - lao động phổ thông
29810,chuyên viên kinh doanh phần mềm,công ty cổ phần phần mềm effect,"ha noi p502, tòa nhà viễn đông, 36 hoàng cầu, p. ô chợ dừa, q. đống đa, tp. hà nội ha noi",ha noi,bán hàng - kinh doanh,toàn thời gian,không,trung cấp,chưa cập nhật,8.5,3.0,7.0,10.0,0,"chuyên viên kinh doanh phần mềm chuyên viên kinh doanh phần mềm tìm kiếm, khai thác khách hàng tiềm năng để tư vấn, giới thiệu sản phẩm/ dịch vụ của công ty. - giao dịch trực tiếp khách hàng, giới thiệu sản phẩm/dịch vụ phần mềm effect. - xúc tiến bán sản phẩm và dịch vụ phần mềm cho khách hàng. - phân tích thị trường, đối thủ cạnh tranh, tìm kiếm và phát triển các mối quan hệ với khách hàng. - thu nhận thông tin phản hồi và chăm sóc khách hàng. yêu cầu công việc: tốt nghiệp cao đẳng, đại học khoa kế toán, tài chính, kinh tế, quản trị kinh doanh. - có khả năng giao tiếp tốt - đam mê và yêu thích công việc kinh doanh. - nhanh nhẹn, sáng tạo, kỹ năng thuyết phục tốt. - có khả năng làm việc độc lập, chịu được áp lực công việc. - ưu tiên các ứng viên có kinh nghiệm trong lĩnh vực bán hàng. có kinh nghiệm về phần mềm kế toán và phần mềm quản trị doanh nghiệp. được hưởng lương cứng theo thỏa thuận về năng lực. mức tối thiểu của nhân viên kinh doanh là 7 triệu – 10 triệu/ tháng. - được cấp laptop. được hưởng nhiều chế độ hỗ trợ và đãi ngộ t

## 5. Lưu bảng kết quả và hình minh họa

In [ ]:
plot_test_cluster_distribution(final_test_clustered, "final_test_cluster_distribution.png")
plot_test_salary_boxplot(final_test_clustered, "final_test_salary_by_cluster_boxplot.png")

demo_columns = [
    "job_title",
    "salary",
    "salary_expected_million_vnd" ,   "job_industry",
    "experience_level",
    "job_position",
    "cluster_id",
    "cluster_label",
]
available_demo_columns = [column for column in demo_columns if column in final_test_clustered.columns]
final_test_demo = final_test_clustered[available_demo_columns].sample(n=min(10, len(final_test_clustered)), random_state=RANDOM_STATE).reset_index(drop=True)
final_test_demo.to_csv(CLUSTERING_DIR / "final_test_demo_10_samples.csv", index=False)
display(final_test_demo)


job_title,salary_expected_million_vnd,job_industry,experience_level,education_level,job_position,job_type,location,location_simplified,cluster_id,cluster_label
kế toán tổng hợp,14.0,kế toán / kiểm toán,4 năm,không,nhân viên,toàn thời gian,"139/9 phan đăng lưu | 141d phan đăng lưu, phường 02, quận phú nhuận, tp hồ chí minh",other,8,Kế toán - kiểm toán - tài chính doanh nghiệp
nhân viên kỹ thuật cơ điện -tại đồng nai (đi làm ngay),9.0,cơ khí - ô tô - tự động hóa / sản xuất - lắp ráp - chế biến,3 năm,trung cấp,nhân viên,toàn thời gian,"lô số 16, đường 10 và 12 kcn giang điền, phường tam phước | kcn biên hòa 1, phường an bình, tp biên hòa, đồng nai",other,1,Kỹ thuật - sản xuất - bảo trì
giám sát an ninh,8.5,chưa xác định,3 năm,trung cấp,nhân viên,toàn thời gian,"phòng 401, tòa nhà ruby tower, số 81-83-83b-85 hàm nghi, phường nguyễn thái bình, quận 1, thành phố hồ chí minh, việt nam",other,6,Bán lẻ - dịch vụ - lao động phổ thông
nhân viên công trường (vệ sinh),8.5,lao động phổ thông,1 năm,không,nhân viên,toàn thời gian,"56 thủ khoa huân, phường bến thành | 193-195 khâm thiên, q.đống đa | quận 1, hồ chí minh",other,10,Xây dựng - kiến trúc - kỹ thuật công trình
trade marketing [quận 10],12.5,marketing,3 năm,đại học,chưa cập nhật,toàn thời gian,"51/2a thành thái, phường diên hồng, thành phố hồ chí minh",other,4,Marketing - nội dung - thiết kế
kế toán trưởng,17.5,quản lý dự án,5 năm,đại học,trưởng phòng,toàn thời gian,"số 9 đường số 10, khu phố 4 , phường hiệp bình chánh, thành phố thủ đức, thành phố hồ chí minh",other,8,Kế toán - kiểm toán - tài chính doanh nghiệp
nhân viên r&d (nghiên cứu phát triển sản phẩm) - nhà máy thực phẩm,11.0,khoa học - kỹ thuật,3 năm,đại học,nhân viên,toàn thời gian,"số 1 đường 4, ktt f361, ngõ 32 phố an dương, phường yên phụ, quận tây hồ, thành phố hà nội, việt nam",other,1,Kỹ thuật - sản xuất - bảo trì
chuyên viên kinh doanh,19.0,bán hàng - kinh doanh,1 năm,cao đẳng,nhân viên,toàn thời gian,"200 đường mê linh, hòa hiệp nam, liên chiểu, đà nẵng | 23/28 nguyên hồng, láng hạ, đống đa, hà nội",other,2,Kinh doanh - bán hàng - chăm sóc khách hàng
nhân viên bảo vệ,6.5,chưa xác định,2 năm,không,nhân viên,toàn thời gian,"53 võ trường toản, phường thảo điền, tp. thủ đức, tp. hcm | 53 võ trường toản - phường thảo điền - quận 2 (hết hiệu lực) - tp hồ chí minh.",ho chi minh,6,Bán lẻ - dịch vụ - lao động phổ thông
"nhân viên bán hàng tại chuỗi cửa hàng thực phẩm tiện lợi satrafoods - gò vấp, bình thạnh",7.0,khách sạn - nhà hàng - du lịch / bán sỉ - bán lẻ - quản lý cửa hàng,2 năm,trung học,nhân viên,toàn thời gian,"455 võ văn tần, phường 05, quận 3, thành phố hồ chí minh",other,6,Bán lẻ - dịch vụ - lao động phổ thông


Đã lưu đầy đủ kết quả gán cụm và các hình minh họa trên tập kiểm thử.


In [8]:
required_outputs = [
    CLEAN_TEST_PATH,
    CLUSTERING_DIR / "final_test_clustered.csv",
    CLUSTERING_DIR / "final_test_demo_10_samples.csv",
    FIGURES_DIR / "final_test_cluster_distribution.png",
    FIGURES_DIR / "final_test_salary_by_cluster_boxplot.png",
]
for path in required_outputs:
    assert path.exists(), f"Thiếu output: {path}"
assert final_test_clustered["cluster_id"].notna().all(), "Thiếu cluster_id trên test."
assert final_test_clustered["cluster_label"].notna().all(), "Thiếu cluster_label trên test."
print("Testing notebook chỉ dùng transform/predict và đã lưu đủ bảng kết quả và hình minh họa.")


Đã lưu đầy đủ kết quả gán cụm và các hình minh họa trên tập kiểm thử.


## Hình minh họa trên tập kiểm thử

![](artifacts/clustering/figures/final_test_cluster_distribution.png)

![](artifacts/clustering/figures/final_test_salary_by_cluster_boxplot.png)
        
